# NB02 — Phase 0: measure compute-need per image

        **~2 hours · shardable across up to 4 accounts · inference only, no training**


> **New here?** Read `05_PLAIN_ENGLISH_GUIDE.md` first — it explains what this
> project is measuring and why, without jargon. This notebook assumes you have.


        ## What we're doing

        We now have four trained models. For each one we ask, **for every one of
        the 10,000 test images**: how little computation is enough?

        Three steps per model:

        **1. Attach early-exit points.** A trained network is one long pipeline.
        We attach five small "read-off points" at 20%, 40%, 60%, 80% and 100% of
        the way through, so we can see what the network would answer if it
        stopped early.

        These read-off points need a few minutes of training themselves — but
        **the main network is frozen solid** while that happens. This is not an
        optimisation, it's the definition: if the main network were allowed to
        adapt, each exit would be reading a *different* network, and "the same
        model with less compute" would stop being true.

        **2. Run every setting on every image.** Five depth settings, five
        resolutions (measured two ways), five numeric precisions — 20 settings ×
        10,000 images per model. We record what the model answered and how
        confident it was, every time.

        No shortcuts here. We can't stop at the first setting that gets the right
        answer, because our definition requires the answer to be *settled* —
        correct at that budget and every larger one. Checking that requires
        actually looking at all of them.

        **3. Also measure the "usual" difficulty scores.** Seven of them, so we
        can later test whether our new measurement is genuinely new or just an
        old idea renamed (that's Q4).

        ## The output

        Two Parquet tables per model: one for the test set, one for a 5,000-image
        slice of training data. **These tables are the actual scientific product
        of this project.** The model weights matter much less — we could throw
        them away and still write the paper.

In [ ]:
# === CELL 1 of every notebook: unpack the library ==========================
# This writes two Python files into the session and imports them. Nothing here
# touches the GPU or the network beyond installing three small packages.
#
#   msc_lib   fdee47a5d80a   the pipeline: HuggingFace sync, model zoo,
#                              measurement, training, the method
#   msc_core  6abdba4ff104   the reference maths: the MSC definition and
#                              every statistic in the paper
#
# Both are generated from KD/src by build_notebooks.py. Editing them HERE does
# nothing useful -- the next rebuild overwrites it. Edit the source instead.
import base64, os, subprocess, sys
from pathlib import Path

WORK = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()

# Kaggle images already ship torch, pandas and sklearn. These three vary by
# image version, so we check rather than assume.
#   pyarrow  writes the per-image measurement tables (Parquet)
#   pynvml   reads GPU power/temperature/utilisation directly
#   fvcore   counts FLOPs, which is how compute cost is defined
for _pkg in ('pyarrow', 'pynvml', 'fvcore', 'psutil'):
    try:
        __import__(_pkg)
    except ImportError:
        print(f'[BOOT] installing {_pkg} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg,
                        '--break-system-packages'], check=False)

_LIB = (
    '...',
)

_CORE = (
    '...',
)

for _name, _blob in (('msc_lib', _LIB), ('msc_core', _CORE)):
    (WORK / f'{_name}.py').write_bytes(base64.b64decode(''.join(_blob)))
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
for _m in [m for m in list(sys.modules) if m in ('msc_lib', 'msc_core')]:
    del sys.modules[_m]

import msc_lib as msc
import msc_core

print(f'[BOOT] msc_lib v{msc.__version__} ready  (torch available: {msc._TORCH_OK})')
print(f'[BOOT] artifact space: {msc.WORK_ROOT}   scratch space: {msc.SCRATCH_ROOT}')

## Step 1 — Session

In [ ]:
# === Who am I? =============================================================
#
# THIS NOTEBOOK: 4 run(s), ~2 GPU-hours total.
# At NUM_WORKERS = 1 that is all of it on this account.
# Raise NUM_WORKERS and run the same notebook on each account
# with a different WORKER_ID to divide it.
#
# ACCOUNT   labels this Kaggle account in the run log. Two accounts calling
#           themselves the same thing makes the log useless.
# WORKER_ID splits the work. Every account runs THE SAME notebook; the only
#           thing that differs is this number. Account 1 -> 0, account 2 -> 1,
#           and so on. Each account then works out, by pure arithmetic, which
#           jobs belong to it -- no communication needed, no chance of two
#           accounts training the same model, no chance of a job being missed.
#
# DEFAULT IS 1: this account does everything in this notebook. That is the
# simplest thing that works. Change it only when you actually have several
# accounts running at once.
# Measurement is inference-only: ~30 min per model.
ACCOUNT     = 'acct1'      # <<< CHANGE ME
NUM_WORKERS = 1          # <<< how many accounts you are running in parallel
WORKER_ID   = 0            # <<< CHANGE ME: 0, 1, 2, ... up to NUM_WORKERS-1

sess = msc.Session(account=ACCOUNT, phase='p0', dataset='cifar100',
                   worker_id=WORKER_ID, num_workers=NUM_WORKERS,
                   shard_mode='cost',        # balance GPU-hours, not job counts
                   enable_hf=True,
                   session_limit_h=8.5,      # push + pause before Kaggle kills us
                   commits_per_hour_limit=20,# 6 accounts x 20 = 120 < HF's 128/hr
                   batch_interval_sec=1800)  # the 30-minute push policy

## Step 2 — Dataset

In [ ]:
# === Get CIFAR-100 =========================================================
# Looks in this order: attached Kaggle dataset (instant) -> earlier extraction
# -> Kaggle CLI download -> torchvision as a last resort.
# Everything lands in /kaggle/temp (~1 TB scratch), never in /kaggle/working
# (20 GB, and that is the space your results need).
DATA_ROOT = sess.prepare_data()
print('dataset:', DATA_ROOT)

## Step 3 — Catch up

In [ ]:
# === Catch up with what has already been done ==============================
# Downloads only what this notebook needs -- never the whole repo, which would
# fill the 20 GB disk instantly.
#
# It also rebuilds the progress record from the actual training logs instead of
# trusting the status file. If a session died between writing a log and pushing
# its status, those two disagree, and the log is the one that tells the truth.
# Runs marked "finished" that clearly are not get reset so they resume.
sess.sync_state(verbose=True)
sess.status()

## Step 4 — Measure

**~30 minutes per model.** Downloads each trained model from HuggingFace
as it needs it, measures, uploads the tables, moves on.

> **If this cell finishes in under a minute, it did nothing.** The
> printed status table before and after tells you directly. A genuine
> run prints exit-head training progress and a sweep progress bar per
> model.

Already measured? It skips instantly — that is the one legitimate way
this is fast. Safe to re-run.

In [ ]:
cfgs = [sess.config(a, seed=s) for a in ('resnet32x4', 'wrn_40_2') for s in (1, 2)]

# Only measure models that actually finished TRAINING.
ready   = [c for c in cfgs if sess.trained(c['run_id'])]
missing = [c['run_id'] for c in cfgs if not sess.trained(c['run_id'])]
if missing:
    print(f'Not yet trained (run NB01 first): {missing}')
    print()

# State BEFORE we start, so it is obvious whether this cell had work to do.
print('measurement status before this run:')
for c in cfgs:
    print(f"  {c['run_id']:34s} trained={sess.trained(c['run_id'])!s:5s} "
          f"measured={sess.measured(c['run_id'])}")
print()

# stage='measure' matters. The ledger already says 'completed' for every run --
# training set that. Without a measurement-specific completion test, this cell
# plans zero work and exits in seconds looking like a success.
results = sess.run_all(ready, fn=sess.oracle, title='Phase 0 measurement',
                       done_fn=sess.measured, stage='measure')
for r in results:
    print(f"{r.get('run_id')}: {r.get('status')}")

print()
print('measurement status after:')
for c in cfgs:
    print(f"  {c['run_id']:34s} measured={sess.measured(c['run_id'])}")

## Step 5 — Are the tables lined up?

Every table must describe the same 10,000 images **in the same order**.
We store a fingerprint of that order in each table and compare.

This check matters enormously. If two tables' rows don't correspond to
the same images, correlating them produces numbers that look completely
plausible and are completely fictional. It's the single easiest way to
accidentally invent a result.

In [ ]:
import pandas as pd
rows = []
for d in sorted(sess.runs_dir.iterdir()) if sess.runs_dir.exists() else []:
    ps = d / 'per_sample'
    if not ps.is_dir():
        continue
    meta = msc.read_json(ps / 'meta.json', default={}) or {}
    for split in ('test', 'train_holdout'):
        f = ps / f'{split}.parquet'
        if f.exists():
            df = pd.read_parquet(f, columns=['sample_order_hash'])
            rows.append({'run_id': d.name, 'split': split, 'rows': len(df),
                         'order_fingerprint': df['sample_order_hash'].iloc[0][:16],
                         'arch': meta.get('arch')})
align = pd.DataFrame(rows)
if not len(align):
    print('No per-sample tables yet.')
    print()
    print('Expected if Step 4 has not produced anything, which happens when the')
    print('backbones are still training. NB02 only measures runs the registry')
    print('marks "completed" -- a half-trained model would give measurements')
    print('that look valid and are not.')
    state = sess.registry.latest()
    prog = [{'run_id': c['run_id'],
             'state': state.get(c['run_id'], {}).get('state', 'not started'),
             'epoch': state.get(c['run_id'], {}).get('epoch', '-')}
            for c in cfgs]
    display(pd.DataFrame(prog))
    print()
    print('Finish NB01 first, then re-run this notebook.')
else:
    display(align)
    for split, g in align.groupby('split'):
        k = g.order_fingerprint.nunique()
        print(f"{split}: {k} distinct ordering(s) -> "
              f"{'OK' if k == 1 else 'MISALIGNED -- DO NOT ANALYSE THESE'}")
    msc.save_analysis(sess.data_dir, 'per_sample_alignment_phase0', align, sess.hub)

## Step 6 — Finish

In [ ]:
# === Push everything and stop ==============================================
# Blocks until HuggingFace confirms. Safe to re-run.
sess.finish()

# D-19: draining the upload queue is NOT the same as the files being on
# HuggingFace, and "[SESSION] done" reads like a confirmation it is not.
# Ask the repository before you close this tab.
#
# D-20: three states, not two. FINISHED and RESUMABLE are both safe -- a run
# paused at epoch 120 whose ckpt_last.pt is on HF loses nothing when you close
# the tab. Only AT RISK (no summary.json AND no checkpoint) needs action.
try:
    _ids = [c['run_id'] for c in (all_cfgs if 'all_cfgs' in dir() else cfgs)]
except NameError:
    _ids = []
if _ids:
    sess.confirm_on_hf(_ids)

---
**Next: NB03** — the go/no-go decision.